In [1]:
import csv
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import pandas as pd
import rasterio
from rasterio.transform import from_origin
import re
import scipy as sp
import pyreadr
import sys
sys.path.append(os.path.abspath('../src/'))
from visualize import *
from utils import *
from scipy import stats
import argparse
import copy
import glob
import argparse
import cProfile
import csv
from functools import partial
import itertools
import math
import matplotlib.pyplot as plt
import multiprocessing as mp
import numpy as np
import os
import rasterio
from scipy import stats
from scipy.optimize import minimize
# from scipy.special import binom
from skimage import graph
import sys
import time
import tqdm
# from make_trap_layout import *
# from simulate_scr import *


Bad key "text.kerning_factor" on line 4 in
/opt/anaconda3/lib/python3.7/site-packages/matplotlib/mpl-data/stylelib/_classic_test_patch.mplstyle.
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.1.3/matplotlibrc.template
or from the matplotlib source distribution


In [2]:
# read in trap locations
trap_coords = pd.read_csv('./data/trap_locations/New_sigma_range_2-27-25_100m_trap_grid_2-27-25.csv')
trap_coords = trap_coords.drop(columns = ['Unnamed: 0'])
trap_coords = trap_coords.rename(columns = {'X': 'x', 'Y': 'y'})
trap_coords

# Create a list of trap coordinates
trap_coords_list = []
for i in range(trap_coords.shape[0]):
    trap_coords_list.append((trap_coords['x'][i], trap_coords['y'][i]))

# Create a list of trap coordinates
trap_coords_list

[(510387.787215289, 5633360.46939679),
 (510487.787215289, 5633360.46939679),
 (510587.787215289, 5633360.46939679),
 (510687.787215289, 5633360.46939679),
 (510787.787215289, 5633360.46939679),
 (510887.787215289, 5633360.46939679),
 (510987.787215289, 5633360.46939679),
 (511087.787215289, 5633360.46939679),
 (511187.787215289, 5633360.46939679),
 (509687.787215289, 5633460.46939679),
 (509787.787215289, 5633460.46939679),
 (509887.787215289, 5633460.46939679),
 (509987.787215289, 5633460.46939679),
 (510087.787215289, 5633460.46939679),
 (510187.787215289, 5633460.46939679),
 (510287.787215289, 5633460.46939679),
 (510387.787215289, 5633460.46939679),
 (510487.787215289, 5633460.46939679),
 (510587.787215289, 5633460.46939679),
 (510687.787215289, 5633460.46939679),
 (510787.787215289, 5633460.46939679),
 (510887.787215289, 5633460.46939679),
 (510987.787215289, 5633460.46939679),
 (511087.787215289, 5633460.46939679),
 (511187.787215289, 5633460.46939679),
 (509187.787215289, 56335

In [3]:
# def find_lcp_to_pts(raster, alpha2, ref_pts, raster_cell_size=1):               # *** What are ref_pts? The activity centers? Seems like these are potential trap locations***
#     """
#     Computes least cost path lengths from every cell in a raster to each of a list of reference points.  
#     """
#     def get_cell_cost(x):
#         return np.exp(x*alpha2)
#     get_all_cell_costs = np.vectorize(get_cell_cost)
#     cost_raster = get_all_cell_costs(raster)
#     lcp_graph = graph.MCP_Geometric(cost_raster, fully_connected=True)

#     lcp_distances = np.zeros((len(ref_pts), raster.shape[0], raster.shape[1]))
#     for rpidx in range(len(ref_pts)):
#         rp = ref_pts[rpidx]
#         rp_int = (int(np.floor(rp[0])), int(np.floor(rp[1])))
#         distances = lcp_graph.find_costs([rp_int])[0]
#         lcp_distances[rpidx, ...] = distances*raster_cell_size

#     return lcp_distances


# def find_euclid_dist(raster, ref_pts, raster_cell_size):
#     """
#     Computes euclidean distance from every cell in a raster to each of a list of reference points.
#     """
#     euclid_distances = np.zeros((len(ref_pts), raster.shape[0], raster.shape[1]))
#     for rpidx in range(len(ref_pts)):
#         rp = ref_pts[rpidx]
#         for i in range(raster.shape[0]):
#             for j in range(raster.shape[1]):
#                 euclid_distances[rpidx, i, j] = np.sqrt((i-rp[0])**2 + (j-rp[1])**2)*raster_cell_size
#     return euclid_distances


def find_euclid_dist_vectorized(raster, ref_pts, raster_cell_size):
    """
    Computes euclidean distance from every cell in a raster to each of a list of reference points.
    """
    x_coords, y_coords = np.meshgrid(np.arange(raster.shape[1]), np.arange(raster.shape[0]))        # Each row in x_coords has the x-coords for all columns. Y_coords has each column contains y_coords for all rows
    euclid_distances = np.zeros((len(ref_pts), raster.shape[0], raster.shape[1]))     # Initiatize distance array with each trap location to have a 2-D array
    
    for rpidx, rp in enumerate(ref_pts):            # Calculate distance to all other cells from a given trap location
        euclid_distances[rpidx] = np.sqrt((x_coords - rp[0])**2 + (y_coords - rp[1])**2) * raster_cell_size
    
    return euclid_distances         # Returns a 2-d map of distances for each reference point (60K x 60K x 60k)

In [4]:
def compute_expected_n(raster, raster_cell_size, distances, trap_locs, g0, sigma, K, density, prob_cap, trap_x):
    """
    Computes expected number of unique individuals detected in a spatial capture-recapture study.
    """

    for t in range(len(trap_locs)):                         # Loop through all possible trap locations
        if int(trap_x[t]) == 0:                             # If a trap is not placed... 
            prob_cap[t,...] = np.zeros((raster.shape[0], raster.shape[1]))  # Probability of capture is 0

    i_cap_hist = np.squeeze(np.zeros((len(trap_locs), 1))) # empty capture history??
    print(i_cap_hist.shape)
    print(prob_cap.shape)
    p_empty_cap_hist = compute_cond_lik_ind(prob_cap, K, len(trap_locs), raster, i_cap_hist)   # Computing inner prodcut in E(n) equation
    p_nonempty = 1 - p_empty_cap_hist     # Computing inner product in E(n) equation part 2
    expected_n = sum(sum(p_nonempty*density))  # density should be a vector indexed by location
    return expected_n

def compute_expected_n_across_scenarios(raster, raster_cell_size, dist, trap_locs, g0, sigma, K, density, trap_x):
    nscenarios = len(g0)
    e_n = np.zeros((nscenarios,1))
    for s in range(nscenarios):
        e_n[s,0] = compute_expected_n(raster[s], raster_cell_size[s], trap_locs, g0[s], sigma[s],  K, density[s], trap_x)
    return(e_n)

In [5]:
def compute_expected_c(raster, raster_cell_size, distances, trap_locs, g0, sigma, K, density, prob_cap, trap_x,):            ### Based on E(r) slide
    """
    Computes expected number of captures.
    """
    for t in range(len(trap_locs)):                                             # Loop through all possible trap locations
        if int(trap_x[t]) == 0:                                                 # If a trap is not placed in that pixel...
            prob_cap[t,...] = np.zeros((raster.shape[0], raster.shape[1]))      # Probability of capture in that pixel is 0

    broadcast_density = np.broadcast_to(density, (len(trap_locs), density.shape[0], density.shape[1]))  # Reshape denisty array so it represents density per pixel in the raster, repeated for each trap location
    expected_c = sum(sum(sum(prob_cap*broadcast_density)))*K                    # Exceptect number of captures = density * x * probs, summed over all k,j, and l.
    return expected_c

def compute_expected_c_across_scenarios(raster, raster_cell_size, dist, trap_locs, g0, sigma, K, density, trap_x):
    nscenarios = len(g0)
    e_c = np.zeros((nscenarios,1))
    for s in range(nscenarios):
        e_c[s,0] = compute_expected_c(raster[s], raster_cell_size[s], dist, trap_locs, g0[s], sigma[s], K, density[s], trap_x)
    return(e_c)

In [6]:
def compute_cond_lik_ind(est_prob_cap, K, num_traps, raster, ind_cap_hist):
    """
    Computes the likelihood of an individual's capture history conditional on their activity center location.
    """
    if raster.ndim == 1:
        # Reshape using est_prob_cap's spatial dimensions
        h, w = est_prob_cap.shape[1], est_prob_cap.shape[2]
        raster = raster.reshape(h, w)

    broadcast_i_cap_hist = np.broadcast_to(ind_cap_hist[..., np.newaxis, np.newaxis], (num_traps, raster.shape[0], raster.shape[1]))        # each traps capture history is repeated across the entire raster grid.
    probs = stats.binom.pmf(broadcast_i_cap_hist, K, est_prob_cap)
    zero_mask = probs == 0.0
    log_probs = np.log(probs, where=np.invert(zero_mask))
    log_probs[zero_mask] = -sys.maxsize - 1
    log_cond_lik_sums = np.sum(log_probs, axis=0)
    return np.exp(log_cond_lik_sums)


In [7]:
def backward_greedy(scenarios, trap_loc, K, distances):
    trap_x = np.ones((len(trap_loc),))                  # initailize all pixels to have a camera trap - binary array representing if trap is activated in that pixel or not.
    landscape_ndarr = []
    raster_cell_size = []
    g0 = []
    sigma = []
    alpha1 = []
    prob_cap = []
    density_prior = []
    E_n_curr = []
    E_c_curr = []
    E_r_curr = []
    RSE_curr = []
    RSE_hist = []

    for s in range(len(scenarios)):                     # for each scenario, calculate the RSE in the event all trap locations are activated
        print(s)
        landscape_ndarr.append(scenarios[s][0])
        raster_cell_size.append(scenarios[s][1])
        g0.append(scenarios[s][2])
        sigma.append(scenarios[s][3])

        # Read in density prior file from /data/density
        density_prior_file =  f'data/density/Dmod_draw_{s+1}.csv'
        density_df = pd.read_csv(density_prior_file)
        density_grid = density_df.pivot_table(values='D_mod',index='x',columns='y', fill_value = 0).astype(np.float64).values.T     # should be size of landscape raster

        # Calculate prob_cap for each scenario
        alpha1.append(1/(2*sigma[s]*sigma[s]))
        prob_cap.append((g0[s])*np.exp(-alpha1[s]*(distances**2)))

        # Compute RSE
        print("computing E_n")
        E_n_curr.append(compute_expected_n(landscape_ndarr[s], raster_cell_size[s], distances, trap_loc, g0[s], sigma[s], K, density_prior[s], prob_cap[s], trap_x))
        print("computing E_c")
        E_c_curr.append(compute_expected_c(landscape_ndarr[s], raster_cell_size[s], distances, trap_loc, g0[s], sigma[s], K, density_prior[s], prob_cap[s], trap_x))
        print("computing E_r")
        E_r_curr.append(E_c_curr[s] - E_n_curr[s])
        print("computing RSE")
        RSE_curr.append(1/np.sqrt(min([E_n_curr[s], E_r_curr[s]])))

    # Average performance across all scenarios
    print("computing average RSE over scenarios")
    RSE_hist.append(np.mean(RSE_curr))
    remove_hist = []
    print(RSE_hist)

    counter = 0
    while sum(trap_x) > 10:    # All camera trap studies have at minimum ~10 traps deployed
        counter += 1           # Number of camera removals
        print(counter)
        trap_indices = [i for i, x in enumerate(trap_x) if int(x) == 1]         # Pixels which are still activated
        # trap_indices = trap_indices[0:10]# truncate early while testing
        E = np.zeros((len(trap_indices), 2))
        trap_x_temp = np.copy(np.broadcast_to(trap_x, (len(trap_indices),trap_x.shape[0])))

        # temporarily remove each currently active location to 0 and simulate the performance
        for pos in range(len(trap_indices)):
            trap_idx = trap_indices[pos]
            trap_x_temp[pos, trap_idx] = 0 

        # Multiprocess the E_n and E_c calculations
        func1 = partial(compute_expected_n_across_scenarios, landscape_ndarr, raster_cell_size, distances, trap_loc, g0, sigma, K, density_prior, prob_cap)
        func2 = partial(compute_expected_c_across_scenarios, landscape_ndarr, raster_cell_size, distances, trap_loc, g0, sigma, K, density_prior, prob_cap)
        pool = mp.Pool(min(mp.cpu_count(), 10))
        E_n_per_scenario = np.squeeze(np.array(pool.map(func1, trap_x_temp)))
        # E[...,0] = np.mean(E_n_per_scenario, axis=1)
        E_c_per_scenario = np.squeeze(np.array(pool.map(func2, trap_x_temp)))
        pool.close()
        pool.join()

        # Calculate RSE
        E_r_per_scenario = E_c_per_scenario
        min_n_r_per_scenario = np.zeros(E_r_per_scenario.shape)
        RSE_per_scenario = np.zeros(E_r_per_scenario.shape)
        for t in range(len(trap_indices)):
            for s in range(len(scenarios)):
                E_r_per_scenario[t,s] -= E_n_per_scenario[t,s]
                min_n_r_per_scenario[t,s] = min(E_n_per_scenario[t,s], E_r_per_scenario[t,s])
                RSE_per_scenario[t,s] = 1/np.sqrt(min_n_r_per_scenario[t,s])
        # E[...,1] = np.mean(E_r_per_scenario, axis=1)
        # min_n_r = np.min(E, axis=1).tolist()
        RSE_temp = np.mean(RSE_per_scenario, axis=1).tolist()
        # RSE_temp = [1/np.sqrt(i) for i in min_n_r]
        # print(RSE_temp)
        # assert 2==3, 'break now'

        # Select which trap to remove based on RSE
        min_change_idx = RSE_temp.index(min(RSE_temp)) 
        remove = trap_indices[min_change_idx]
        trap_x[remove] = 0

        # Track removal history
        RSE_hist.append(RSE_temp[min_change_idx])
        remove_hist.append(remove)
        print(remove, RSE_temp[min_change_idx])
    
    return(remove_hist, RSE_hist)


In [18]:
# Read in parameter draws
params = pd.read_csv('data/params/New_sigma_range_2-27-25_param_values_for_each_draw300_2-27-25.csv')
params = params.rename(columns = {'Unnamed: 0': 'index'})

# Extract parameter values
D = [params['D'][i] for i in range(params.shape[0])]
g0 = [params['g0'][i] for i in range(params.shape[0])]
sigma = [params['sigma'][i] for i in range(params.shape[0])]
raster_cell_size = [100 for i in range(params.shape[0])]
beta1 = [params['beta1'][i] for i in range(params.shape[0])]        # beta parameters not needed directly for optimization
beta2 = [params['beta2'][i] for i in range(params.shape[0])]
beta3 = [params['beta3'][i] for i in range(params.shape[0])]
beta4 = [params['beta4'][i] for i in range(params.shape[0])]

# Define estimated N and K values
N = 47      # Estimates based on 2019 study -- Not needed in current format since we are NOT using uniform density
K = 5       # Number of sampling periods

# Read in landscape raster
landscape_ndarr = []        # Initialize an empty list to store the raster data as numpy arrays
landscape_raster = rasterio.open('data/raster/New_sigma_range_2-27-25_100m_mask_2-27-25.tif')
scenario_landscape_ndarr = np.squeeze(np.array(landscape_raster.read()))    # reads in the raster data into memory and converts into an array
landscape_ndarr.append(scenario_landscape_ndarr)        # appends the 2d numpy array (rows,columns) representation of the raster to the list
landscape_shape = landscape_ndarr[0].size           # define size of the raster (# rows x # columns)


# Read in trap locations (consistent across all scenarios and currently all pixels)
trap_loc = []
trap_loc_path = 'data/trap_locations/New_sigma_range_2-27-25_100m_trap_grid_2-27-25.csv'
with open(trap_loc_path, 'r') as f:
    reader = csv.reader(f)
    header = next(reader)  # Extract header row
    x_idx = header.index('x')  # Get index of 'x' column
    y_idx = header.index('y')  # Get index of 'y' column
    scenario_trap_loc = [(eval(row[x_idx]), eval(row[y_idx])) for row in reader]
trap_loc.append(scenario_trap_loc)

# Calculate uniform density priors -- update this to account for the density priorsn needs to happen in the for loop 
# density_prior = []
# density_prior.append(np.ones((scenario_landscape_ndarr.shape[0], scenario_landscape_ndarr.shape[1]))*(N/float(scenario_landscape_ndarr.size))) 

# Define scenarios
scenarios = list(zip(*[landscape_ndarr, raster_cell_size, g0, sigma]))
trap_loc = trap_loc[0] # trap locations are the same for all scenarios

## TESTING DATA - COMMENT OUT WHEN NOT IN USE
# landscape_data = landscape_ndarr[0]
# subset_raster = landscape_data[:100, :100]
# subset_traps = np.array(trap_loc[:5])
# print(subset_raster.shape)
# print(subset_raster.size)
# test_dist = find_euclid_dist_vectorized(landscape_data[:1000,:1000], subset_traps, 100)
# density_prior = []
# density_prior.append(np.ones((100, 100))*(N/subset_raster.size))
# scenarios = list(zip(*[subset_raster, raster_cell_size, g0, sigma, density_prior, ch_index]))


# Calculate the overall distances between all points in the landscape and the trap locations
overall_dist = find_euclid_dist_vectorized(landscape_ndarr[0], np.array(trap_loc), 100)
fp = np.memmap('data/overall_dist.dat', dtype=np.float64, mode='w+', shape=overall_dist.shape)
fp[:] = overall_dist  # Save data
# fp.flush()  # Ensure data is written to disk
# # # Load data
# loaded_dist = np.memmap('data/overall_dist.dat', dtype=np.float64, mode='r', shape = (len(trap_coords_list), landscape_raster.shape[0], landscape_raster.shape[1]))

# # removed_traps, RSE_trace = backward_greedy(scenarios, trap_loc, K)
# backward_greedy(scenarios, trap_loc, K, loaded_dist)
# backward_greedy(scenarios, subset_traps, K, test_dist)

(510487.787215289, 5633360.46939679)


In [9]:
# Convert to float32 upfront
# landscape_data = landscape_ndarr[0].astype(np.float32)
# trap_coords = np.array(trap_loc, dtype=np.float32)

# chunk_size = 500  # Adjust based on available RAM
# for i in range(0, len(trap_coords), chunk_size):
#     chunk = trap_coords[i:i+chunk_size]
#     dist_chunk = find_euclid_dist_vectorized(landscape_data, chunk, 100)
#     # Save chunk to disk (e.g., .npy files)


In [10]:
# Read in parameter draws
params = pd.read_csv('data/params/New_sigma_range_2-27-25_param_values_for_each_draw300_2-27-25.csv')
params = params.rename(columns={'Unnamed: 0': 'index'})

# Extract parameter values for ALL scenarios
D = params['D'].tolist()
g0 = params['g0'].tolist()
sigma = params['sigma'].tolist()
raster_cell_size = [100] * len(params)
ch_index = params.index.tolist()

# Define estimated N and K values
N = 47
K = 5

# Read in landscape raster
landscape_raster = rasterio.open('data/raster/New_sigma_range_2-27-25_100m_mask_2-27-25.tif')
scenario_landscape_ndarr = landscape_raster.read(1)  # Read first band as 2D array

# Read in trap locations
trap_loc_path = 'data/trap_locations/New_sigma_range_2-27-25_100m_trap_grid_2-27-25.csv'
trap_loc = pd.read_csv(trap_loc_path)[['x', 'y']].values

# Function to load density files
def load_density(scenario_id):
    """Load density file and convert to 2D grid matching raster dimensions"""
    density_path = f'data/density/New_sigma_range_2-27-25_D_mod_Dmod_draw_{scenario_id}.csv'
    density_df = pd.read_csv(density_path)
    
    # Convert to grid format using lat/long as coordinates
    density_grid = density_df.pivot_table(
        values='D_mod',
        index='x',
        columns='y',
        fill_value=0
    ).values
    
    # Match raster dimensions
    if density_grid.shape != scenario_landscape_ndarr.shape:
        density_grid = np.resize(density_grid, scenario_landscape_ndarr.shape)
    
    return density_grid

# Configure scenarios (index 0=id1, index9=id10)
selected_indices = [0, 9]
scenarios = []

for idx in selected_indices:
    # Get parameters for this scenario
    scenario_params = {
        'raster': scenario_landscape_ndarr,
        'cell_size': 100,
        'g0': g0[idx],
        'sigma': sigma[idx],
        'density': load_density(idx+1),  # id=index+1
        'ch_index': ch_index[idx]
    }
    scenarios.append(scenario_params)

# TESTING WITH SUBSET (100x100) --------------------------------------------
test_raster = scenario_landscape_ndarr[:100, :100]
test_traps = trap_loc[:5]

# Load subset density (first 100x100 pixels of original density grid)
for scenario in scenarios:
    scenario['raster'] = test_raster
    scenario['density'] = scenario['density'][:100, :100]

# Calculate distances for subset
test_dist = find_euclid_dist_vectorized(test_raster, test_traps, 100)

# Convert scenarios to tuple list format expected by backward_greedy
scenario_tuples = [
    (
        s['raster'], 
        s['cell_size'], 
        s['g0'], 
        s['sigma'], 
        s['density'], 
        s['ch_index']
    ) for s in scenarios
]

# Run analysis on selected scenarios
backward_greedy(scenario_tuples, test_traps, K, test_dist)


FileNotFoundError: [Errno 2] No such file or directory: 'data/density/New_sigma_range_2-27-25_D_mod_Dmod_draw_1.csv'